**Programmer: python_scripts (Abhijith Warrier)**

PYTHON SCRIPT TO ***BUILD A VECTOR DATABASE FOR FAST AND EFFICIENT SEMANTIC SEARCH USING PYTHON***. 🔍🧠📚

Large Language Models are powerful at generating responses, but they don't automatically have access to private documents, company knowledge bases, or recently updated information.

**Retrieval-Augmented Generation (RAG)** addresses this limitation by retrieving relevant information from external knowledge sources and providing it to an LLM as additional context.

But how do we efficiently search thousands or millions of documents to find the most relevant information?

This is where **Vector Stores** become essential.

In this project, we'll build a simple vector store using **FAISS (Facebook AI Similarity Search)** and Sentence Transformers. We'll convert text into embeddings, store them in a FAISS index, and retrieve relevant documents using semantic similarity.

---

## 📦 Install Required Libraries

We'll use FAISS for vector similarity search, Sentence Transformers for generating embeddings, and NumPy for handling numerical arrays.

In [ ]:
pip install faiss-cpu sentence-transformers numpy

---

## 📦 Import Required Libraries

In [ ]:
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

FAISS provides efficient similarity-search algorithms, while Sentence Transformers converts text into dense numerical vectors that represent semantic meaning.

---

## 🧠 Step 1 — Understand Vector Embeddings

Traditional keyword search looks for exact or similar words within documents. Semantic search takes a different approach by comparing the meaning of text. For example, consider these two sentences:

**Sentence A:** "How can I learn Python programming?"

**Sentence B:** "What is the best way to get started with Python?"

Although they use different words, both express a similar intent. Embedding models represent sentences as numerical vectors in a multidimensional space. Texts with similar meanings generally have embeddings that are closer together.

A simplified embedding might look like:

```
"Learn Python" → [0.21, -0.45, 0.78, 0.12, ...]
```

These vectors can then be indexed and searched efficiently.

---

## 🤖 Step 2 — Load an Embedding Model

We'll use a lightweight Sentence Transformer model to generate document embeddings.

In [ ]:
embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

The `all-MiniLM-L6-v2` model produces **384-dimensional embeddings**, making it suitable for lightweight semantic search demonstrations. The same embedding model should be used for both documents and search queries to ensure they share a compatible vector space.

---

## 📚 Step 3 — Create a Document Collection

Let's create a small knowledge base containing information about Python, Machine Learning, and AI.

In [ ]:
documents = [
    "Python is a versatile programming language used for web development, automation, and data science.",
    "Machine Learning allows computers to identify patterns and learn from data.",
    "Deep Learning uses artificial neural networks with multiple layers.",
    "Natural Language Processing enables computers to understand and generate human language.",
    "Retrieval-Augmented Generation combines information retrieval with language model generation.",
    "Vector databases store numerical embeddings for efficient similarity searches.",
    "FAISS is a library developed by Meta for efficient similarity search over dense vectors.",
    "Sentence Transformers generates embeddings that represent the semantic meaning of text.",
    "Large Language Models can generate text, answer questions, and summarize information.",
    "Semantic search retrieves information based on meaning rather than exact keyword matches."
]

In real-world RAG applications, these documents might come from PDFs, websites, internal documentation, databases, or other knowledge sources. For simplicity, we'll use a predefined collection of text documents.

---

## 🔢 Step 4 — Generate Document Embeddings

Next, we'll convert our documents into numerical embeddings.

In [ ]:
document_embeddings = embedding_model.encode(documents, convert_to_numpy=True)

document_embeddings = np.asarray(document_embeddings, dtype="float32")

print("Embedding Shape:", document_embeddings.shape)

Since we have 10 documents and each embedding contains 384 dimensions, the expected shape is:

```
Embedding Shape: (10, 384)
```

Each row represents one document, while each column represents a dimension of its embedding. These embeddings will become the searchable data inside our FAISS index.

---

## 🗂️ Step 5 — Build a FAISS Vector Index

FAISS provides several indexing strategies for similarity search. For this project, we'll use `IndexFlatIP`, which performs exact nearest-neighbor search using inner-product similarity. To make inner-product search equivalent to cosine similarity, we'll normaliSe the embeddings first.

In [ ]:
faiss.normalize_L2(document_embeddings)
dimension = document_embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(document_embeddings)

print("Total Indexed Documents:", index.ntotal)

**What's happening here?**

- `normalize_L2()` converts embeddings into unit-length vectors.
- `IndexFlatIP()` creates an index that compares vectors using inner products.
- `index.add()` inserts our document embeddings into the index.
- `index.ntotal` returns the total number of indexed vectors.

Because our vectors are normalised, their inner products correspond to cosine similarity scores. For a small dataset, `IndexFlatIP` is a straightforward choice because it performs exact searches without requiring index training.

---

## 🔍 Step 6 — Perform Semantic Search

Now that our vector store is ready, we can search it using natural language. Suppose a user asks:

```
query = "How does RAG help language models access external information?"
```

First, convert the query into an embedding.

In [ ]:
query_embedding = embedding_model.encode([query], convert_to_numpy=True)
query_embedding = np.asarray(query_embedding, dtype="float32")

faiss.normalize_L2(query_embedding)

The query embedding must have the same dimensionality as the stored document embeddings.

Next, search for the three most similar documents.

In [ ]:
top_k = 3

scores, indices = index.search(query_embedding, top_k)

FAISS returns two arrays:

- `scores` contains similarity scores.
- `indices` contains the positions of matching documents.

We can use these indices to retrieve the original text.

---

## 📊 Step 7 — Display Retrieved Documents

Let's display the documents ranked by semantic similarity.

In [ ]:
print("\nSearch Query:", query)
print("\nTop Matching Documents:")

for rank, (doc_index, score) in enumerate(
    zip(indices[0], scores[0]),
    start=1
):
    print(f"\nRank {rank}")
    print(f"Document: {documents[doc_index]}")
    print(f"Similarity Score: {score:.4f}")

A possible result might look like:

```
Search Query: How does RAG help language models access external information?

Top Matching Documents:

Rank 1
Document: Retrieval-Augmented Generation combines information retrieval with language model generation.
Similarity Score: 0.72

Rank 2
Document: Large Language Models can generate text, answer questions, and summarize information.
Similarity Score: 0.43

Rank 3
Document: Vector databases store numerical embeddings for efficient similarity searches.
Similarity Score: 0.38
```

*The scores and ranking above are illustrative. Actual results depend on the embedding model and query.* Notice that semantic search doesn't require the query to contain exactly the same words as the retrieved document. Instead, the embedding model helps FAISS identify documents that are semantically related to the query.

---

## ⚙️ Step 8 — Create a Reusable Search Function

Rather than repeating the embedding and retrieval steps for every query, we can create a reusable function.

In [ ]:
def semantic_search(query, top_k=3):
    query_vector = embedding_model.encode([query], convert_to_numpy=True)

    query_vector = np.asarray(query_vector, dtype="float32")
    faiss.normalize_L2(query_vector)

    scores, indices = index.search(query_vector, min(top_k, index.ntotal))

    results = []

    for doc_index, score in zip(indices[0], scores[0]):
        results.append({
            "document": documents[doc_index],
            "similarity": float(score)
        })

    return results

Now we can search our knowledge base using different queries.

In [ ]:
results = semantic_search("What is semantic search?", top_k=3)

for result in results:
    print(result)

This function returns the most relevant documents along with their similarity scores.

---

## 💾 Step 9 — Save and Reload the FAISS Index

Generating embeddings and building an index can be expensive when working with large document collections.

Instead of rebuilding the index every time the application starts, FAISS allows us to save it to disk.

In [ ]:
faiss.write_index(index, "knowledge_base.index")

print("FAISS index saved successfully!")

We can later reload the index:

In [ ]:
loaded_index = faiss.read_index("knowledge_base.index")

print("Loaded Index Size:", loaded_index.ntotal)

This makes the vector index reusable across application sessions.

**Important:** FAISS stores vectors, not the original document text. In a real application, document content and metadata must also be persisted separately so retrieved vector IDs can be mapped back to their source documents.

Additionally, only load FAISS index files from trusted sources.

---

## 🔗 Step 10 — Understand How FAISS Fits into RAG

We've now built the retrieval component of a RAG system. A complete RAG workflow generally looks like this:

```
              DOCUMENT INGESTION
                     │
                     ▼
              Source Documents
                     │
                     ▼
              Text Chunking
                     │
                     ▼
              Embedding Model
                     │
                     ▼
               FAISS Index
                     │
                     │
              USER QUERY
                     │
                     ▼
              Query Embedding
                     │
                     ▼
              Similarity Search
                     │
                     ▼
              Relevant Documents
                     │
                     ▼
              Context + User Query
                     │
                     ▼
              Language Model
                     │
                     ▼
              Generated Answer
```

FAISS handles the similarity-search stage, retrieving relevant information from the indexed knowledge base. The retrieved documents can then be included in a prompt sent to an LLM, allowing it to generate a response grounded in external information.

---

## 🧩 FAISS vs Traditional Databases

FAISS is a similarity-search library rather than a complete database management system.

| Feature | FAISS | Traditional Relational Database |
| --- | --- | --- |
| Primary Purpose | Vector similarity search | Structured data storage and querying |
| Search Approach | Nearest-neighbor search | SQL queries, indexes, filters |
| Data Representation | Numerical vectors | Tables, rows, and columns |
| Semantic Search | Supported through embeddings | Requires additional vector capabilities |
| Document Metadata | Managed separately | Supported through database schemas |
| Common Applications | RAG, semantic search, recommendations | Business applications, transactions, reporting |

Modern databases such as PostgreSQL can also support vector search through extensions like pgvector, so the distinction isn't always absolute. FAISS is particularly useful when developers want direct control over vector indexing and similarity-search algorithms.

---

## 🌍 Real-World Applications

FAISS-based vector stores can support several AI applications.

**1. Document Question Answering:** Retrieve relevant sections from PDFs, manuals, and internal documentation before generating answers.

**2. Enterprise Knowledge Search:** Find information across large collections of organizational documents using natural-language queries.

**3. AI Customer Support:** Retrieve relevant troubleshooting instructions and knowledge-base articles.

**4. Recommendation Systems:** Identify similar products, articles, or other items based on embedding similarity.

**5. Research Assistants:** Retrieve relevant information from collections of research papers and technical documentation.

**6. RAG Pipelines:** Provide relevant external context to language models to improve the grounding of generated responses.

---

## 🧠 Key Takeaways

- Vector stores enable efficient similarity search over numerical embeddings.
- Sentence Transformers converts text into dense vectors representing semantic meaning.
- FAISS provides efficient nearest-neighbor search over vector collections.
- `IndexFlatIP` supports exact inner-product search.
- L2 normalization allows inner-product search to represent cosine similarity.
- Query embeddings must use a compatible embedding model and vector dimensions.
- FAISS indexes can be saved and reloaded for reuse.
- Original documents and metadata must be maintained separately from the vector index.
- FAISS provides the retrieval foundation for building RAG applications.

---

## Conclusion

Building a FAISS Vector Store is an important step toward understanding how modern Retrieval-Augmented Generation systems retrieve information.

In this project, we explored how to convert documents into embeddings using Sentence Transformers, organize those embeddings inside a FAISS index, and perform semantic similarity searches to retrieve relevant information. We also learned how to create reusable search functions, persist vector indexes, and understand the relationship between vector retrieval and language generation.

While FAISS doesn't generate answers itself, it provides an efficient way to identify relevant information that can later be supplied to an LLM as context.

**By combining embeddings with vector similarity search, we establish the foundation for building document search engines, knowledge assistants, and more advanced RAG applications.**

---